## Lab 2: Neural Network Basics
***(Suggested time: 30-40 minutes)***

Accelerator : **T4 GPU**


This lab takes us through fundamentals of Neural Networks and Keras APIs.  It uses many ideas from the [Introduction to Keras for Engineers](https://keras.io/getting_started/intro_to_keras_for_engineers/) webpage.

The goal of this lab is to show you:
- The setup needed to train a neural network model
- Use of Keras compile method to choose optimizer and loss function
- Invoke Keras **fit** method for training
- How to load an existing dataset from Keras
- Example of a model created using Keras sequential and functional approach


##### **Step 0.** 
Setup Keras backend and print its version.  Keras backend can be set to either one of the following: jax, torch or tensorflow.  By default it is set to tensorflow.



In [0]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
print ('Keras version is', keras.__version__)

##### **Step 1:**  
Train a simple linear regression model using Keras APIs.

The simplest regression model has one neuron and fits equation of a line y = mx + b. Here *m* is the slope (weight) and *b* is the offset (bias).


In [0]:
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf
import numpy as np
import keras
from keras import layers
from keras import models

# Create a simple Keras model. Expected output is 2x-1
# The output is obtained using list comprehension, output = [2*x-1 for x in input]
# You can also try a different linear equation of your choice
input = [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
output = [-3, -1, 1, 3, 5, 7, 9, 11, 13,15]

X = np.array(input)
y = np.array(output)

tf.keras.backend.clear_session()

model = keras.Sequential()
model.add(keras.Input(shape=(1,)))
model.add(layers.Dense(1))


model.compile(optimizer='sgd', loss='mean_squared_error')
model.fit(X, y, epochs=200)

print (model.summary())

weights = model.get_weights()
print (weights)

**Experiment:** As expected the weight is close to 2.0 and bias is close to -1.0. Change the X and y lists to represent a different linear sequence and see if the model is able to correctly determine the slope and bias.

Run training for another 200 training steps to see if there is an improvement in results.

In [0]:
model.fit(X, y, epochs=200)

print (model.summary())

weights = model.get_weights()

print (weights)

##### Question: 
Why are the slope/offset values obtained after evoking fit() a second time closer to the expected value than the first time?

#### Solution

<details>
      <summary> Click here to see our answer </summary>

       This is because the model training uses the existing start and builds upon it to get better results.     
</details>



##### Question:   
Is this simplest neural network model deterministic?


#### Solution

<details> 
    <summary> Click here to see our answer </summary>
    
         The model is **not** deterministic. The values of the neuron weight and bias change each time the model is trained.

</details>
      


##### Question: 
Why is mean squared error (mse) used as loss function for the above sequence?

#### Solution

<details> 
    <summary>Click here to see our answer </summary>

    This is because we are trying to predict next value/values in a sequence or series. Often referred to as regression.

</details>     


##### **Step 2.** 
Load and preprocess MNIST dataset of handwritten numbers.
Learn more about the MNIST [dataset](https://www.tensorflow.org/datasets/catalog/mnist) and know other available dataset.  
- Use the MNIST dataset to understand details such as input size, number of classes, number of training/test samples and finally see some examples from the dataset.  
- Make sure the details from the dataset match the output from the cell below.
- Each MNIST image is a black & white image of 28x28 pixels

In [0]:
# Load the data and split it between train and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Scale images to the [0, 1] range
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)


print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")

##### **Step 3:** 
Create an AI model using CNN layers for classification of MNIST digits
- Convolutional layers are used to extract features from MNIST images

In [0]:
# Model parameters
num_classes = 10
input_shape = (28, 28, 1)

model = keras.Sequential(
    [
        keras.layers.Input(shape=input_shape),
        keras.layers.Conv2D(8, kernel_size=(3, 3), activation="relu"),
        keras.layers.Conv2D(16, kernel_size=(3, 3), activation="relu"),
        keras.layers.MaxPooling2D(pool_size=(2, 2)),
        keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(num_classes, activation="softmax"),
    ]
)

model.summary()

##### Experiment: 
- Try changing the number of 2D convolution layers in Step 3 and see how it affects number of parameters in the model
- Why does the GlobalAveragePooling layer has no trainable parameters?

#### Solution

<details open>
  <summary>Click here to see our answer</summary>

Global pooling has no trainable parameters because it is an arithmetic operation.

</details>



##### **Step 4:** Configure model training to use Sparse Categorical Crossentropy, Adam Optimizer, and accuracy metric.

In [0]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="acc"),
    ],
)

##### **Step 5:** Train a model using .fit method.

- Set a batch_size of 128 and train the model for 20 Epochs.
- Choose early stopping callback to stop the training if the model validation loss values stops improving.
- Configure Validation split to be 20% to check model performance during training
- Check model performance using .evaluate method on the test dataset

The training process will take about 3 to 5 minutes when running on a CPU.



In [0]:
batch_size = 128
epochs = 5

callbacks = [
     keras.callbacks.EarlyStopping(monitor="val_loss", patience=2),
]

model.fit(
    x_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.20,
    callbacks=callbacks,
)
score = model.evaluate(x_test, y_test, verbose=0)

print ("Test loss:", score[0])
print ("Test accuracy:", score[1])

##### **Step 6:** Save and load Keras models

- Save the trained model as a Keras(.keras) model
- Clear the model from working memory
- Load the saved Keras model and then have it predict using the test dataset

In [0]:
model.save("mnist_model.keras")

keras.backend.clear_session()

model = keras.models.load_model("mnist_model.keras")

predictions = model.predict(x_test)

print(predictions)


##### Question: 
What is the expected shape of the predictions array? Explain the values in the two nested arrays that you see after printing out predictions.  



#### Solution

<details open>
  <summary>Click here to see our answer</summary>

    - The expected shape of predictions is (10000, 10). It isknown by print(predictions.shape)
    - The nested array consists of 10000 rows as we have 10000 test data in the MNIST dataset
    - There are 10 columns in each row.  Each column value denotes the probability of a value from 0 to 9.
    

</details>

##### **Step 7:** Create a model with non-sequential layers.

- Use of Functional API to create a model with non-sequential layers. The functional approach allows for complex architectures such as shared layers, multiple inputs, and multiple outputs.
- The sample code below represents sensor fusion using inputs from two sensors: branches called as input_a and input_b.
- In the code, we call each layer with an input tensor. The call processes that tensor and returns a new output tensor
- Two layers are merged using concatenate layer in this example.
- The resulting model architecture of two inputs and one output is saved to an image which captures layer names and shape.



In [0]:
from tensorflow import keras
from keras.layers import Input, Dense, concatenate
from keras.models import Model

# Define two input layers - one from each sensor
input_tensor_a = Input(shape=(32,), name='input_a')
input_tensor_b = Input(shape=(32,), name='input_b')

# Define layers and connect them to sensor inputs
# Branch for input_a (sensor a)
x = Dense(16, activation='relu')(input_tensor_a)
x = Dense(8, activation='relu')(x)

# Branch for input_b (sensor b)
y = Dense(16, activation='relu')(input_tensor_b)
y = Dense(8, activation='relu')(y)

# Concatenate the outputs of the two branches
merged = concatenate([x, y], name='merged_layer')

# Final output layer
main_output = Dense(1, activation='sigmoid', name='main_output')(merged)

# Create the Model by specifying inputs and outputs
model = Model(inputs=[input_tensor_a, input_tensor_b], outputs=main_output)

model.summary()

keras.utils.plot_model(model, "two_input_one_output.png", show_shapes=True, show_layer_names=True)

# 4. Compile the model (for a single output)
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

##### Viewing generated model architecture
- The model architecture is converted to a diagram using *keras.utils.plot_model* utility. 
- This architecture (.png) file is shown below. 

In [0]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread("two_input_one_output.png")
plt.figure(figsize=(12, 12))
plt.imshow(img)
plt.axis("off")
plt.show()

##### Question:
What does batch size none signify in the model architecture diagram or in the model summary?

#### Solution

<details open>
  <summary>Click here to see our answer</summary>
This is set to None as we do not want to hardcode the training to a strict batch size. Setting it to None allows us to train with 32 images at once during one step, 16 images at another, and then evaluate with a single image in production without having to rebuild the model.
</details>

##### Experiment:
Change the model architecture to take three inputs and provide two outputs.

*Hint:* The final Model API takes inputs and outputs as a list.

### Exercise 1

Using the above cell as an example; load and print details of the CIFAR 10 dataset.  You can find details about the CIFAR10 dataset [here](https://www.tensorflow.org/datasets/catalog/cifar10).  Keras command to load dataset is mentioned [here](https://keras.io/api/datasets/cifar10/).

- Scale the pixels of the training and test data to a range of 0 to 1.
- Expand dimensions of training and test data to include number of samples
- Print number of samples in training and test sections

#### Solution to Exercise 1

<details> 
    <summary> Click here to see our answer </summary>

    (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

    # Scale images to the [0, 1] range
    x_train = x_train.astype("float32") / 255
    x_test = x_test.astype("float32") / 255
    # Make sure images have shape (28, 28, 1)
    x_train = np.expand_dims(x_train, -1)
    x_test = np.expand_dims(x_test, -1)
    print("x_train shape:", x_train.shape)
    print("y_train shape:", y_train.shape)
    print(x_train.shape[0], "train samples")
    print(x_test.shape[0], "test samples")

</details>
      



##### Question:  What is the shape of the input image in the CIFAR10 dataset?

#### Solution

<details open>
  <summary>Click here to see our answer</summary>

The input shape for CIFAR10 is (32,32,3).  This represents image dimensions (32x32 pixels) and 3 channels (RGB color).

</details>

##### Exercise 2

Load and preprocess the CIFAR-100 dataset. Create a CNN architecture and train it to classify 100 classes of CIFAR-100 dataset.

- Scale the pixels of the training and test data to a range of 0 to 1.
- Print number of samples in training and test sections
- *Comment on the model accuracy!*
- The purpose of the exercise is to extend our learning of model classification from 10 to 100 classes. 

##### Solution to Exercise 2

<details>
    <summary> Click here for our answer </summary>

    import tensorflow as tf
    from tensorflow import keras
    from keras.datasets import cifar100
    from keras.utils import to_categorical

    # Load CIFAR-100 dataset
    (x_train, y_train), (x_test, y_test) = cifar100.load_data()

    # Normalize pixel values
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0

    # One-hot encode labels
    num_classes = 100
    y_train = to_categorical(y_train, num_classes)
    y_test = to_categorical(y_test, num_classes)

    model = keras.Sequential([
        keras.layers.Input(shape=(32, 32, 3)),
        keras.layers.Conv2D(32, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(128, (3, 3), activation='relu'),
        keras.layers.GlobalAveragePooling2D(),
            keras.layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

    history = model.fit(x_train, y_train,
                    epochs=10,
                    batch_size=64,
                    validation_data=(x_test, y_test))

    loss, accuracy = model.evaluate(x_test, y_test)
    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")

   
</details>

##### Reasons for low accuracy and suggestions for improvement

<details>
    <summary> Click here for our answer </summary>

    Reasons:
    - Model Capacity vs. Task Complexity (Our model is underfitting. Not enough resources to learn features of 100 classes)
    - Information Loss from Over-Pooling (Two layers of max pooling and one layer of global pooling)
    - Training duration is quite low (You can see the validation accuracy is still improving!)

    Suggestions:    
    - Increase Epochs: Move from 10 to at least 20.
    - BatchNormalization
    - Transfer Learning: If you want 70%+ accuracy quickly, use a pre-trained model like ResNet50V2 or EfficientNet, 
      (we will discuss this in a subsequent module) 
    
</details>